In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from sklearn.metrics import log_loss
from src.models import whiff

wm = whiff.build()
print(wm.X_train.shape, wm.X_validation.shape, "cal split at", wm.n_calibration)

(200735, 31) (83492, 31) cal split at 41746


In [2]:
eval_part = wm.split.validation.iloc[wm.n_calibration:].copy()
eval_part["y"] = eval_part["target"].to_numpy()
eval_part["p_model"] = wm.predict(eval_part)
eval_part["p_base"] = wm.predict_baseline(eval_part)

print(f"eval rows: {len(eval_part):,}")
print(f"model    log loss: {log_loss(eval_part['y'], eval_part['p_model']):.5f}")
print(f"baseline log loss: {log_loss(eval_part['y'], eval_part['p_base']):.5f}")

eval rows: 41,746
model    log loss: 0.46449
baseline log loss: 0.48102


In [3]:
def group_loss(data, by, min_n=500):
    rows = []
    for key, g in data.groupby(by, observed=True):
        if len(g) < min_n:
            continue
        rows.append({
            "group": key,
            "n": len(g),
            "actual": g["y"].mean(),
            "model": log_loss(g["y"], g["p_model"], labels=[0, 1]),
            "baseline": log_loss(g["y"], g["p_base"], labels=[0, 1]),
        })
    out = pd.DataFrame(rows)
    out["improvement"] = out["baseline"] - out["model"]
    return out.sort_values("improvement")

pt = eval_part["pitch_type"].astype(str)
eval_part["pitch_type_b"] = pt.where(pt.isin(wm.keep_types), "OTHER")

print("=== by pitch type ===")
print(group_loss(eval_part, "pitch_type_b").round(4).to_string(index=False))

=== by pitch type ===
group     n  actual  model  baseline  improvement
   KC   769  0.3277 0.5782    0.5457      -0.0325
   FC  3661  0.1950 0.4888    0.4639      -0.0250
   SI  5966  0.1056 0.3352    0.3275      -0.0077
   FF 13391  0.1784 0.4367    0.4564       0.0197
   ST  2973  0.2994 0.5171    0.5384       0.0213
   SL  5990  0.3309 0.5146    0.5392       0.0246
   CH  4735  0.3010 0.5438    0.5787       0.0349
   FS  1464  0.3108 0.5312    0.5752       0.0440
   CU  2428  0.3138 0.4856    0.5399       0.0543


In [4]:
resid = eval_part.groupby("pitcher").agg(
    n=("y", "size"), actual=("y", "mean"), predicted=("p_model", "mean"))
resid = resid[resid["n"] >= 100]
resid["bias"] = resid["actual"] - resid["predicted"]

print(f"{len(resid)} pitchers with 100+ swings in eval")
print(resid["bias"].describe().round(4).to_string())

145 pitchers with 100+ swings in eval
count    145.0000
mean      -0.0123
std        0.0431
min       -0.1426
25%       -0.0381
50%       -0.0143
75%        0.0115
max        0.1044


In [5]:
# 1. Does the model lose where the height effect is weak?
height_corr = {}
for pt, g in wm.split.train.groupby("pitch_type"):
    if len(g) < 2000:
        continue
    height_corr[pt] = pd.to_numeric(g["plate_z_rel"], errors="coerce").corr(g["target"])

gl = group_loss(eval_part, "pitch_type_b")
gl["height_corr"] = gl["group"].map(height_corr)
print(gl[["group", "n", "improvement", "height_corr"]].round(4).to_string(index=False))
print()
print("improvement vs |height_corr|:",
      round(gl["improvement"].corr(gl["height_corr"].abs()), 3))

group     n  improvement  height_corr
   KC   769      -0.0325      -0.5008
   FC  3661      -0.0250      -0.0844
   SI  5966      -0.0077       0.0058
   FF 13391       0.0197       0.2066
   ST  2973       0.0213      -0.3257
   SL  5990       0.0246      -0.3766
   CH  4735       0.0349      -0.3170
   FS  1464       0.0440      -0.4127
   CU  2428       0.0543      -0.4458

improvement vs |height_corr|: 0.385


In [6]:
# 2. How much headroom is there in pitcher identity?
best_possible = eval_part.merge(
    resid[["bias"]], left_on="pitcher", right_index=True, how="left")
best_possible["p_adj"] = (best_possible["p_model"] + best_possible["bias"].fillna(0)).clip(0.001, 0.999)

print()
print("current model:        ", round(log_loss(eval_part["y"], eval_part["p_model"]), 5))
print("with pitcher offsets: ", round(log_loss(best_possible["y"], best_possible["p_adj"]), 5))


current model:         0.46449
with pitcher offsets:  0.46024
